In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from qaoa_rw import data, qubo, qaoa, baselines, metrics

RESULTS, FIGS = pathlib.Path("../results"), pathlib.Path("../figures")
RESULTS.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
N_HW, K_HW = 6, 3
P_LIST = [1, 2]
SHOTS = 4000

RUN_ON_HARDWARE = False
BUDGET_SECONDS = 9 * 60
RESERVE_SECONDS = 90
BACKEND_NAME = None


In [ ]:
Xh, yh, names_h, rel_h, red_h = data.load_problem(N_HW, seed=SEED)
Qh, off_h = qubo.feature_selection_qubo(rel_h, red_h, K_HW, 0.5, 2.0)
Eh = qubo.energies(Qh, off_h)
h, J, const = qubo.qubo_to_ising(Qh, off_h)
print("Óptimo exacto:", [n for n, b in zip(names_h, qubo.index_to_mask(int(np.argmin(Eh)), N_HW)) if b])

trained, prev = {}, None
for p in P_LIST:
    inits = [qaoa.interp_init(prev["params"])] if prev is not None else None
    r = qaoa.run_qaoa(Eh, N_HW, p, restarts=15, seed=SEED + p, x0_list=inits)
    trained[p], prev = r, r

ideal = {p: metrics.summarize(Eh, r["prob"], N_HW, K_HW) for p, r in trained.items()}
display(pd.DataFrame(ideal).T.round(4))

np.savez(RESULTS / "hw_problem.npz", Q=Qh, offset=off_h, n=N_HW, k=K_HW)
json.dump({str(p): r["params"].tolist() for p, r in trained.items()}, open(RESULTS / "hw_ideal.json", "w"), indent=1)


In [ ]:
from qaoa_rw import ibm
from qiskit.quantum_info import Statevector

for p, r in trained.items():
    ops = qaoa.circuit_ops(h, J, r["params"])
    assert np.allclose(np.abs(qaoa.run_ops_numpy(ops, N_HW)) ** 2, r["prob"], atol=1e-8), "ops != statevector"
    qc = ibm.build_circuit(ops, N_HW, measure=False)
    assert np.allclose(Statevector(qc).probabilities(), r["prob"], atol=1e-8), "Qiskit != statevector"
    print(f"p={p}: circuito verificado ✔")


In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
fake = FakeSherbrooke()

rows = []
for n in (4, 6, 8):
    _, _, _, rn, dn = data.load_problem(n, SEED)
    Qn, on = qubo.feature_selection_qubo(rn, dn, max(2, n // 2))
    hn, Jn, _ = qubo.qubo_to_ising(Qn, on)
    for p in (1, 2):
        isa = ibm.transpile_for(fake, ibm.build_circuit(qaoa.circuit_ops(hn, Jn, np.full(2 * p, 0.3)), n))
        rows.append({"qubits": n, "p": p, **ibm.circuit_stats(isa)})
display(pd.DataFrame(rows))


In [ ]:
from qiskit_aer import AerSimulator
sim = AerSimulator.from_backend(fake)

noisy_counts, noisy = {}, {}
for p, r in trained.items():
    isa = ibm.transpile_for(fake, ibm.build_circuit(qaoa.circuit_ops(h, J, r["params"]), N_HW))
    counts = sim.run(isa, shots=SHOTS).result().get_counts()
    noisy_counts[p] = counts
    noisy[p] = metrics.summarize(Eh, ibm.counts_to_prob(counts, N_HW), N_HW, K_HW)
display(pd.DataFrame(noisy).T.round(4))
json.dump({str(p): c for p, c in noisy_counts.items()}, open(RESULTS / "hw_noisy_sim.json", "w"))


In [ ]:
import datetime as dt

if RUN_ON_HARDWARE:
    service = ibm.get_service()
    backend = service.backend(BACKEND_NAME) if BACKEND_NAME else service.least_busy(
        operational=True, simulator=False, min_num_qubits=N_HW)
    print("Backend:", backend.name)

    spent = 0.0
    for p, r in trained.items():
        if BUDGET_SECONDS - spent < RESERVE_SECONDS:
            print("Presupuesto casi agotado: me detengo."); break
        isa = ibm.transpile_for(backend, ibm.build_circuit(qaoa.circuit_ops(h, J, r["params"]), N_HW))
        print(f"p={p}:", ibm.circuit_stats(isa))
        job = ibm.run_sampler(backend, [isa], shots=SHOTS,
                              max_execution_time=min(120, int((BUDGET_SECONDS - spent) / 2)))
        counts = job.result()[0].data.meas.get_counts()
        used = ibm.job_usage_seconds(job)
        spent += used if used is not None else 60.0
        json.dump({"backend": backend.name, "job_id": job.job_id(), "n": N_HW, "k": K_HW, "p": p,
                   "params": r["params"].tolist(), "shots": SHOTS, "counts": counts, "usage_seconds": used,
                   "timestamp": dt.datetime.now().isoformat(timespec="seconds")},
                  open(RESULTS / f"ibm_{backend.name}_n{N_HW}_p{p}.json", "w"))
        print(f"p={p}: job {job.job_id()} · uso {used} s · acumulado ~{spent:.0f} s de {BUDGET_SECONDS}")
else:
    print("RUN_ON_HARDWARE=False: no se gastó tiempo de IBM.")


In [ ]:
import glob
for f in sorted(glob.glob(str(RESULTS / "ibm_*_n*_p*.json"))):
    d = json.load(open(f))
    s = metrics.summarize(Eh, ibm.counts_to_prob(d["counts"], d["n"]), d["n"], d["k"])
    print(pathlib.Path(f).name, {k: round(v, 4) for k, v in s.items()})
